# Phase 1.5 — Build Medical Knowledge Base (BM25 Retrieval Index)

**Goal:** Extract unique medical contexts from the full ViMedAQA dataset to create a searchable knowledge base using BM25.

**Input:** `../data/raw/vimedaq_full.json` (created in Phase 1)

**Output:**
- `../data/processed/medical_corpus.json` — list of unique medical contexts
- `../data/processed/bm25_index.pkl` — serialized BM25Okapi index

**Runtime:** CPU (no GPU needed)  
**Estimated time:** < 5 minutes  
**⚠️ Prerequisite:** Phase 1 must be completed (`vimedaq_full.json` must exist).

## Cell 1 — Setup & Verify Prerequisites

In [1]:
import os
import json
import pickle

# --- Path Configuration ---
# Adjust DATA_ROOT depending on your environment:
#   Colab:  "/content/drive/MyDrive/vimedaq-project"
#   Local:  "../data" (relative to this notebook)
DATA_ROOT = "../data"  # Change this if running on Colab with Drive

RAW_DATA_PATH     = os.path.join(DATA_ROOT, "raw", "vimedaq_full.json")
CORPUS_OUT_PATH   = os.path.join(DATA_ROOT, "processed", "medical_corpus.json")
BM25_OUT_PATH     = os.path.join(DATA_ROOT, "processed", "bm25_index.pkl")

# Prerequisite check: Phase 1 must have produced vimedaq_full.json
if not os.path.exists(RAW_DATA_PATH):
    raise FileNotFoundError(
        f"⚠️ PREREQUISITE FAILED: '{RAW_DATA_PATH}' not found.\n"
        f"You must complete Phase 1 (01_data_exploration.ipynb) first.\n"
        f"If running on Colab, make sure Google Drive is mounted and DATA_ROOT is correct."
    )

print(f"✅ Raw data found: {RAW_DATA_PATH}")
print(f"   Corpus output:  {CORPUS_OUT_PATH}")
print(f"   BM25 output:    {BM25_OUT_PATH}")

✅ Raw data found: ../data\raw\vimedaq_full.json
   Corpus output:  ../data\processed\medical_corpus.json
   BM25 output:    ../data\processed\bm25_index.pkl


## Cell 2 — Load Raw Data & Extract Unique Contexts

In [2]:
# Load the full dataset (all splits combined, saved in Phase 1)
with open(RAW_DATA_PATH, "r", encoding="utf-8") as f:
    all_data = json.load(f)

print(f"Total records loaded: {len(all_data)}")
print(f"Sample keys: {list(all_data[0].keys())}")

# Extract unique contexts to build the medical corpus
# Using a set to deduplicate, then convert to sorted list for deterministic ordering
context_set = set()
for item in all_data:
    ctx = item.get("context", "").strip()
    if ctx:  # Skip empty contexts
        context_set.add(ctx)

corpus = sorted(list(context_set))  # Sort for reproducibility

print(f"\n--- Corpus Statistics ---")
print(f"Total records in dataset:  {len(all_data)}")
print(f"Unique contexts (corpus):  {len(corpus)}")
print(f"Deduplication ratio:       {len(corpus)/len(all_data)*100:.1f}%")
print(f"\nFirst context preview (truncated):")
print(f"  '{corpus[0][:120]}...'")

Total records loaded: 44313
Sample keys: ['question_idx', 'question', 'answer', 'context', 'title', 'keyword', 'topic', 'article_url', 'author', 'author_url', '_original_split']

--- Corpus Statistics ---
Total records in dataset:  44313
Unique contexts (corpus):  17955
Deduplication ratio:       40.5%

First context preview (truncated):
  '(Cụ thể là chứng nôn, tiêu chảy, ra mồ hôi nhiều, chân tay lạnh, mạch nhỏ khó bắt). Phụ tử chế 12 g, Can khương 10 g, Ch...'


## Cell 3 — Build BM25 Index

In [3]:
from rank_bm25 import BM25Okapi

# Tokenize corpus using basic whitespace splitting
# This is sufficient for Vietnamese BM25 (word-level matching)
tokenized_corpus = [doc.split() for doc in corpus]

print(f"Building BM25 index over {len(tokenized_corpus)} documents...")
bm25 = BM25Okapi(tokenized_corpus)
print(f"✅ BM25 index built successfully.")
print(f"   Vocabulary size (approx): {len(bm25.idf)} terms")

Building BM25 index over 17955 documents...
✅ BM25 index built successfully.
   Vocabulary size (approx): 47081 terms


## Cell 4 — Save Corpus & BM25 Index

In [4]:
# Ensure output directory exists
os.makedirs(os.path.dirname(CORPUS_OUT_PATH), exist_ok=True)

# Save corpus as JSON (human-readable, inspectable)
with open(CORPUS_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)
print(f"✅ Corpus saved: {CORPUS_OUT_PATH} ({os.path.getsize(CORPUS_OUT_PATH)/1e6:.1f} MB)")

# Save BM25 index as pickle (binary, fast to reload)
with open(BM25_OUT_PATH, "wb") as f:
    pickle.dump(bm25, f)
print(f"✅ BM25 index saved: {BM25_OUT_PATH} ({os.path.getsize(BM25_OUT_PATH)/1e6:.1f} MB)")

✅ Corpus saved: ../data\processed\medical_corpus.json (12.0 MB)
✅ BM25 index saved: ../data\processed\bm25_index.pkl (15.0 MB)


## Cell 5 — Validation: Test Retrieval Quality

In [5]:
# Quick validation: query the BM25 index with sample questions
test_questions = [
    "Triệu chứng của bệnh tiểu đường là gì?",
    "Thuốc Paracetamol có tác dụng gì?",
    "Gan có chức năng gì trong cơ thể?",
]

print("=" * 70)
print("BM25 Retrieval Validation")
print("=" * 70)

for q in test_questions:
    tokenized_query = q.split()
    scores = bm25.get_scores(tokenized_query)
    top_idx = scores.argsort()[-1]  # Top-1 index
    top_score = scores[top_idx]
    retrieved_context = corpus[top_idx]

    print(f"\n🔍 Question: {q}")
    print(f"   Score:    {top_score:.4f}")
    print(f"   Context:  {retrieved_context[:150]}...")
    print("-" * 70)

print("\n✅ Retrieval validation complete. Review contexts above for relevance.")

BM25 Retrieval Validation

🔍 Question: Triệu chứng của bệnh tiểu đường là gì?
   Score:    17.2962
   Context:  - Triệu chứng của tôi có phải do bệnh Lyme gây ra không?
- Các nguyên nhân khác có thể gây ra triệu chứng của tôi là gì?
- Tôi cần làm xét nghiệm gì?
...
----------------------------------------------------------------------

🔍 Question: Thuốc Paracetamol có tác dụng gì?
   Score:    18.4882
   Context:  N-Acetylcystein là dẫn chất N – acetyl của L-Cystein, một amino acid tự nhiên. Hoạt chất Acetylcystein có tác dụng làm loãng nhầy. Thuốc làm giảm độ đ...
----------------------------------------------------------------------

🔍 Question: Gan có chức năng gì trong cơ thể?
   Score:    20.7670
   Context:  Gan à cơ quan lớn nhất của bụng. Gan có chức năng tổng hợp protein và giải độc máu....
----------------------------------------------------------------------

✅ Retrieval validation complete. Review contexts above for relevance.


## Cell 6 — Reload Validation (simulates a fresh session)

Verify that saved files can be reloaded correctly (important for app.py).

In [6]:
# Simulate fresh load (as app.py will do)
with open(CORPUS_OUT_PATH, "r", encoding="utf-8") as f:
    reloaded_corpus = json.load(f)

with open(BM25_OUT_PATH, "rb") as f:
    reloaded_bm25 = pickle.load(f)

# Verify consistency
assert len(reloaded_corpus) == len(corpus), "Corpus length mismatch!"
test_q = "bệnh tiểu đường".split()
orig_scores = bm25.get_scores(test_q)
reload_scores = reloaded_bm25.get_scores(test_q)
assert (orig_scores == reload_scores).all(), "BM25 scores mismatch!"

print(f"✅ Reload validation passed.")
print(f"   Corpus: {len(reloaded_corpus)} documents")
print(f"   BM25 index: functional")
print(f"\n🎉 Phase 1.5 complete. Files ready for Phase 7 (Web App).")

✅ Reload validation passed.
   Corpus: 17955 documents
   BM25 index: functional

🎉 Phase 1.5 complete. Files ready for Phase 7 (Web App).
